# 21 — Data Visualization

**ENAI 603 Capstone — WMATA Metro Delay Prediction**

This notebook provides map visualizations of WMATA rail stations and line predictions from data collected.

**Sections:**
1. Load WMATA datasets: Predictions and Stations
2. Map WMATA Rail System network using Folium 

## 1. Load WMATA datasets: Predictions and Stations

In [1]:
import sqlite3
from pathlib import Path

import pandas as pd
import numpy as np

# Paths
PROJECT_DIR = Path("..")
DB_PATH = PROJECT_DIR / "data" / "wmata.db"

conn = sqlite3.connect(DB_PATH)
print(f"Connected to {DB_PATH}")
print(f"Database size: {DB_PATH.stat().st_size / 1024:.0f} KB")

Connected to ../data/wmata.db
Database size: 56156 KB


In [2]:
# Load predictions dataset
df_pred = pd.read_sql("SELECT * FROM predictions", conn)

# Convert dates to DateTime format
df_pred["collected_at"] = pd.to_datetime(df_pred["collected_at"], utc=True)

# Parse minutes: numeric → int, ARR/BRD → 0, others → NaN
df_pred["minutes_num"] = pd.to_numeric(df_pred["minutes"], errors="coerce")
df_pred.loc[df_pred["minutes"].isin(["ARR", "BRD"]), "minutes_num"] = 0

# Drop non-informative rows (---, --, empty, No)
df_pred = df_pred.dropna(subset=["minutes_num"]).copy()

# Keep only valid Metro lines
df_pred = df_pred[df_pred["line"].isin(["RD", "BL", "GR", "YL", "OR", "SV"])].copy()
df_pred

print(f"Date range: {df_pred['collected_at'].min()} to {df_pred['collected_at'].max()}")

df_pred

Date range: 2026-03-11 02:00:05.155939+00:00 to 2026-03-18 18:30:16.166019+00:00


,id,collected_at,car,destination,destination_code,destination_name,group_num,line,location_code,location_name,minutes,minutes_num
0,1,2026-03-11 02:00:05.155939+00:00,8,Glenmont,B11,Glenmont,1,RD,A02,Farragut North,BRD,0.0
1,2,2026-03-11 02:00:05.155939+00:00,8,Glenmont,B11,Glenmont,1,RD,A03,Dupont Circle,BRD,0.0
2,3,2026-03-11 02:00:05.155939+00:00,8,Shady Grv,NaN,Shady Grv,2,RD,A04,Woodley Park-Zoo/Adams Morgan,BRD,0.0
3,4,2026-03-11 02:00:05.155939+00:00,6,Glenmont,B11,Glenmont,1,RD,A05,Cleveland Park,BRD,0.0
4,5,2026-03-11 02:00:05.155939+00:00,6,Glenmont,B11,Glenmont,1,RD,A08,Friendship Heights,BRD,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...
339624,339625,2026-03-18 18:30:16.166019+00:00,8,Ashburn,N12,Ashburn,2,SV,N09,Innovation Center,32,32.0
339625,339626,2026-03-18 18:30:16.166019+00:00,6,Ashburn,N12,Ashburn,2,SV,N04,Spring Hill,34,34.0
339626,339627,2026-03-18 18:30:16.166019+00:00,8,Ashburn,N12,Ashburn,2,SV,N10,Washington Dulles International Airport,36,36.0
339627,339628,2026-03-18 18:30:16.166019+00:00,6,Franconia,J03,Franconia-Springfield,2,BL,C06,Arlington Cemetery,37,37.0


In [3]:
# Load stations dataset
df_stations = pd.read_sql("SELECT * FROM stations", conn)
print(f"Total stations: {len(df_stations)}")
df_stations

Total stations: 102


,station_code,station_name,lat,lon,line_code1,line_code2,line_code3,line_code4,together_station
0,A01,Metro Center,38.898303,-77.028099,RD,NaN,NaN,None,C01
1,A02,Farragut North,38.903192,-77.039766,RD,NaN,NaN,None,
2,A03,Dupont Circle,38.909499,-77.043620,RD,NaN,NaN,None,
3,A04,Woodley Park-Zoo/Adams Morgan,38.924999,-77.052648,RD,NaN,NaN,None,
4,A05,Cleveland Park,38.934703,-77.058226,RD,NaN,NaN,None,
...,...,...,...,...,...,...,...,...,...
97,N08,Herndon,38.952821,-77.385178,SV,NaN,NaN,None,
98,N09,Innovation Center,38.960758,-77.415295,SV,NaN,NaN,None,
99,N10,Washington Dulles International Airport,38.955784,-77.448148,SV,NaN,NaN,None,
100,N11,Loudoun Gateway,38.992040,-77.460685,SV,NaN,NaN,None,


---

## 2. Map WMATA Rail System network using Folium

In [4]:
RAIL_MAP = {
  "RD": ['shady', 'rockville', 'twinbrook', 'north bethesda', 'grosvenor', 'medical', 'bethesda', 'friendship', 'tenleytown', 'van ness', 'cleveland', 'woodley', 'dupont', 'farragut north', 'metro',
         'gallery', 'judiciary', 'union', 'noma', 'rhode', 'brookland', 'fort', 'takoma', 'silver', 'forest', 'wheaton', 'glenmont'],
  "BL": ['franconia', 'van dorn', 'king', 'braddock', 'potomac yard', 'reagan', 'crystal', 'pentagon city', 'pentagon', 'arlington', 'rosslyn', 'foggy', 'farragut west', 'mcpherson', 'metro',
         'federal triangle', 'smithsonian', 'enfant', 'federal center', 'capitol south', 'eastern', 'potomac ave', 'stadium', 'benning', 'capitol heights', 'addison', 'morgan boulevard', 'largo'],
  "GR": ['branch', 'suitland', 'naylor', 'southern', 'congress', 'anacostia', 'ballpark', 'waterfront', 'enfant', 'archives', 'gallery',
           'vernon', 'shaw', 'cardozo', 'columbia', 'georgia', 'fort', 'west hyattsville', 'hyattsville crossing', 'college', 'greenbelt'],
  "YL": [['huntington', 'eisenhower', 'king', 'braddock', 'potomac yard', 'reagan', 'crystal', 'pentagon city', 'pentagon', 'enfant', 'archives', 'gallery',
         'vernon', 'shaw', 'cardozo', 'columbia', 'georgia', 'fort', 'west hyattsville', 'hyattsville crossing', 'college', 'greenbelt'],
         ['huntington', 'eisenhower', 'king', 'braddock', 'potomac yard', 'reagan', 'crystal', 'pentagon city', 'pentagon', 'enfant', 'archives', 'gallery',
         'vernon']],
  "OR": ['vienna', 'dunn', 'west falls', 'east falls', 'ballston', 'virginia', 'clarendon', 'court', 'rosslyn', 'foggy', 'farragut west', 'mcpherson', 'metro',
         'federal triangle', 'smithsonian', 'enfant', 'federal center', 'capitol south', 'eastern', 'potomac ave', 'stadium', 'minnesota', 'deanwood', 'cheverly', 'landover', 'carrollton'],
  "SV": [['ashburn', 'loudoun', 'dulles', 'innovation', 'herndon', 'reston town', 'wiehle', 'spring hill', 'greensboro', 'tysons', 'mclean', 'east falls', 'ballston', 'virginia', 'clarendon', 'court', 'rosslyn', 'foggy', 'farragut west', 'mcpherson', 'metro',
         'federal triangle', 'smithsonian', 'enfant', 'federal center', 'capitol south', 'eastern', 'potomac ave', 'stadium', 'minnesota', 'deanwood', 'cheverly', 'landover', 'carrollton'],
         ['ashburn', 'loudoun', 'dulles', 'innovation', 'herndon', 'reston town', 'wiehle', 'spring hill', 'greensboro', 'tysons', 'mclean', 'east falls', 'ballston', 'virginia', 'clarendon', 'court', 'rosslyn', 'foggy', 'farragut west', 'mcpherson', 'metro',
         'federal triangle', 'smithsonian', 'enfant', 'federal center', 'capitol south', 'eastern', 'potomac ave', 'stadium', 'benning', 'capitol heights', 'addison', 'morgan boulevard', 'largo']]
}

In [5]:
import folium
import folium.plugins as plugins
from folium.plugins import HeatMap
from folium.plugins import OverlappingMarkerSpiderfier
from fuzzywuzzy import process

def flatten(arr): # Flatten an array (list)
  flat_list = []
  for row in arr:
      flat_list.extend(row)
  return flat_list

def unique(arr): # Extract unique values from an array (list)
  return list(dict.fromkeys(arr))

# Extract Metro Center station coordinates from WMATA Stations dataset
lat, lon = df_stations.loc[df_stations['station_name'] == 'Metro Center', ['lat', 'lon']].iloc[0]

# Create a Folium map centered on Metro Center station
m = folium.Map(
  location = [lat, lon],
  zoom_start = 11
)

# Add Markers for each station with popup info
station_names = list(df_stations['station_name'])
line_colors = {"RD": "red", "BL": "blue", "OR": "orange", "SV": "silver",
               "GR": "lightgreen", "YL": "yellow"}

for line, stations in RAIL_MAP.items():
  # Handle lines with +2 terminals
  if line in ["YL", "SV"]:
    stations = unique(flatten(stations))
    
  for stop in stations:
    best_match = process.extractOne(stop, station_names)
    station = df_stations[df_stations['station_name'] == best_match[0]].iloc[0]

    popup_html = f"""
    <b>Station {station['station_code']}:</b><br>
    [{line}] {station['station_name']}
    """

    color = line_colors[line]
    folium.Marker(
        location=[station['lat'], station['lon']],
        popup=folium.Popup(popup_html, max_width=200),
        icon=plugins.BeautifyIcon(
                   icon="train",
                   icon_shape="circle",
                   background_color=color
               ),
        tooltip=f"<b>Click to see lines at</b><br>Station {station['station_code']}: {station['station_name']}"
    ).add_to(m)
      
# Manage lines sharing station stops (overlapping markers) by "spiderfying" them when clicked
oms = OverlappingMarkerSpiderfier(
    keep_spiderfied=True,  # Markers remain spiderfied after clicking
    nearby_distance=20,  # Distance for clustering markers in pixel
    circle_spiral_switchover=10,  # Threshold for switching between circle and spiral
    leg_weight=2.0  # Line thickness for spider legs
    )
oms.add_to(m)

m